In [1]:
!mkdir -p ~/.kaggle && echo KGAT_e382b66919744406c252810e86733f1e > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [2]:
!export KAGGLE_API_TOKEN=KGAT_302c4bd2575c0d83b1a45d1947f902e2 # Copied from the settings UI

In [3]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
!kaggle competitions download -c beyond-visible-spectrum-ai-for-agriculture-2026
!unzip beyond-visible-spectrum-ai-for-agriculture-2026.zip

100% 191M/191M [00:01<00:00, 164MB/s]

Archive:  beyond-visible-spectrum-ai-for-agriculture-2026.zip
  inflating: Kaggle_Prepared/train/HS/Health_hyper_1.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_10.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_100.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_101.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_102.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_103.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_104.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_105.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_106.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_107.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_108.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_109.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_11.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hyper_110.tif  
  inflating: Kaggle_Prepared/train/HS/Health_hype

In [4]:
import torch
import torch.nn as nn
import timm  # مكتبة الموديلات الجاهزة

def create_baseline_model():

    model = timm.create_model('resnet50', pretrained=True, num_classes=3)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_baseline_model().to(device)

print(f" model ready on : {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

الموديل جاهز وشغال على: cuda


In [5]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
import time

def train_model(model, train_loader, criterion, optimizer, num_epochs=5):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100. * correct / total
        end_time = time.time()

        print(f'Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}% - Time: {end_time - start_time:.2f}s')



In [7]:
import pandas as pd
import os
import re

# Path where the Kaggle data is unzipped
kaggle_prepared_dir = '/content/Kaggle_Prepared'

csv_paths = !find {kaggle_prepared_dir} -maxdepth 2 -name "*.csv"

if not csv_paths:
    print("No CSV files found. Generating 'train_labels.csv' from training image filenames.")

    generated_data = []
    train_base_path = os.path.join(kaggle_prepared_dir, 'train')

    for modality in ['HS', 'MS', 'RGB']:
        modality_path = os.path.join(train_base_path, modality)
        if os.path.isdir(modality_path):
            for filename in os.listdir(modality_path):
                file_id = os.path.splitext(filename)[0]
                match = re.match(r'^(Health|Rust|Other)_.*', file_id)
                if match:
                    label = match.group(1)
                    generated_data.append({'file_name': file_id, 'label': label, 'split': 'train', 'modality': modality})

    if generated_data:
        labels_df = pd.DataFrame(generated_data)
        output_csv_path = os.path.join(kaggle_prepared_dir, 'train_labels.csv')
        labels_df.to_csv(output_csv_path, index=False)
        csv_path = output_csv_path #
        print(f"Successfully generated '{os.path.basename(csv_path)}' at {os.path.dirname(csv_path)}")
    else:

        raise FileNotFoundError("No CSV files found and unable to generate a 'train_labels.csv' from image filenames with discernible labels.")
else:
    csv_path = csv_paths[0]

print(f"تم إيجاد الملف في: {csv_path}")

# قراءة الملف
df = pd.read_csv(csv_path)
print(f"number of photos: {len(df)}")
print(df.head())
print("Classes")
print(df['label'].value_counts())

No CSV files found. Generating 'train_labels.csv' from training image filenames.
Successfully generated 'train_labels.csv' at /content/Kaggle_Prepared
تم إيجاد الملف في: /content/Kaggle_Prepared/train_labels.csv
number of photos: 1800
          file_name   label  split modality
0     Rust_hyper_37    Rust  train       HS
1   Other_hyper_169   Other  train       HS
2  Health_hyper_153  Health  train       HS
3   Other_hyper_109   Other  train       HS
4  Health_hyper_189  Health  train       HS
Classes
label
Rust      600
Other     600
Health    600
Name: count, dtype: int64


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import os

class WheatDataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.transform = transform
        self.label_map = {"Health": 0, "Rust": 1, "Other": 2}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.base_dir, row['split'], 'RGB', row['file_name'] + '.png')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        label = torch.tensor(self.label_map[row['label']])

        return image, label

train_ds = WheatDataset(df, '/content/Kaggle_Prepared')
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

In [10]:
import timm
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model('resnet50', pretrained=True, num_classes=3)
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

print(f" model ready for trainning on  : {device}")

الموديل جاهز للتدريب على: cuda


In [11]:
num_epochs = 10

print("trainning start")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f} - Acc: {acc:.2f}%')



🚀 بدأ التدريب...
Epoch [1/10] - Loss: 1.0840 - Acc: 41.56%
Epoch [2/10] - Loss: 1.0148 - Acc: 56.72%
Epoch [3/10] - Loss: 0.9010 - Acc: 64.33%
Epoch [4/10] - Loss: 0.7718 - Acc: 68.67%
Epoch [5/10] - Loss: 0.6450 - Acc: 74.56%
Epoch [6/10] - Loss: 0.5259 - Acc: 80.17%
Epoch [7/10] - Loss: 0.4189 - Acc: 85.22%
Epoch [8/10] - Loss: 0.3276 - Acc: 89.11%
Epoch [9/10] - Loss: 0.2560 - Acc: 92.39%
Epoch [10/10] - Loss: 0.2181 - Acc: 92.39%
✅ خلصنا أول Baseline!


In [12]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_ds = WheatDataset(train_df, '/content/Kaggle_Prepared')
val_ds = WheatDataset(val_df, '/content/Kaggle_Prepared')

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print("val images number" ,len(val_df))

تم تجهيز الـ val_loader بنجاح! عدد صور التحقق: 360


In [13]:
from sklearn.metrics import classification_report
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=["Health", "Rust", "Other"]))

              precision    recall  f1-score   support

      Health       0.95      0.94      0.95       120
        Rust       0.96      1.00      0.98       120
       Other       0.96      0.93      0.94       120

    accuracy                           0.96       360
   macro avg       0.96      0.96      0.96       360
weighted avg       0.96      0.96      0.96       360



In [14]:
import tifffile as tiff
import numpy as np
import torch.nn.functional as F

class MultimodalWheatDataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.label_map = {"Health": 0, "Rust": 1, "Other": 2}
        self.target_hs_channels = 126

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rgb_path = os.path.join(self.base_dir, row['split'], 'RGB', row['file_name'] + '.png')
        rgb_img = cv2.imread(rgb_path)
        rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)
        rgb_tensor = torch.from_numpy(rgb_img).permute(2, 0, 1).float() / 255.0

        hs_path = os.path.join(self.base_dir, row['split'], 'HS', row['file_name'] + '.tif')
        hs_img = tiff.imread(hs_path)

        if hs_img.ndim == 2:
            hs_img = np.expand_dims(hs_img, axis=-1)

        hs_tensor = torch.from_numpy(hs_img).permute(2, 0, 1).float()

        current_hs_channels = hs_tensor.shape[0]
        if current_hs_channels < self.target_hs_channels:
            padding_needed = self.target_hs_channels - current_hs_channels
            hs_tensor = F.pad(hs_tensor, (0, 0, 0, 0, 0, padding_needed), 'constant', 0)
        elif current_hs_channels > self.target_hs_channels:
            hs_tensor = hs_tensor[:self.target_hs_channels, :, :]

        label = torch.tensor(self.label_map[row['label']])

        return rgb_tensor, hs_tensor, label

In [15]:
import torch.nn as nn
import timm

class MultimodalFusionModel(nn.Module):
    def __init__(self, hs_channels, num_classes=3):
        super(MultimodalFusionModel, self).__init__()

        self.rgb_branch = timm.create_model('resnet50', pretrained=True, num_classes=0) # بدون آخر طبقة

        self.hs_branch = nn.Sequential(
            nn.Conv2d(hs_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.classifier = nn.Linear(2048 + 64, num_classes)

    def forward(self, rgb, hs):
        rgb_features = self.rgb_branch(rgb)
        hs_features = self.hs_branch(hs)

        combined_features = torch.cat((rgb_features, hs_features), dim=1)

        output = self.classifier(combined_features)
        return output

model_innovative = MultimodalFusionModel(hs_channels=126).to(device)
optimizer = torch.optim.Adam(model_innovative.parameters(), lr=0.0001)


✅ تم بناء موديل الابتكار بنجاح!


In [16]:
sample_dataset = MultimodalWheatDataset(df, '/content/Kaggle_Prepared')
rgb_sample, hs_sample, label_sample = sample_dataset[0]
print(f"RGB Image Shape: {rgb_sample.shape}")
print(f"HS Image Shape: {hs_sample.shape}")

RGB Image Shape: torch.Size([3, 64, 64])
HS Image Shape: torch.Size([126, 32, 32])


In [17]:
import torch.nn as nn
import timm

class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes=3):
        super(MultimodalFusionModel, self).__init__()

        self.rgb_branch = timm.create_model('resnet50', pretrained=True, num_classes=0)

        self.hs_branch = nn.Sequential(
            nn.Conv2d(126, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.classifier = nn.Linear(2048 + 64, num_classes)

    def forward(self, rgb, hs):
        rgb_feat = self.rgb_branch(rgb)
        hs_feat = self.hs_branch(hs)

        combined = torch.cat((rgb_feat, hs_feat), dim=1)
        return self.classifier(combined)


model_innovative = MultimodalFusionModel().to(device)
optimizer = torch.optim.Adam(model_innovative.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

In [18]:
def train_multimodal(model, loader, epochs=10):
    model.train()
    for epoch in range(epochs):
        correct = 0
        total = 0
        for rgb, hs, labels in loader:
            rgb, hs, labels = rgb.to(device), hs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(rgb, hs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        print(f"Epoch [{epoch+1}/{epochs}] - Accuracy: {100.*correct/total:.2f}%")


train_loader_multi = DataLoader(MultimodalWheatDataset(train_df, '/content/Kaggle_Prepared'), batch_size=16, shuffle=True)
train_multimodal(model_innovative, train_loader_multi)

Epoch [1/10] - Accuracy: 53.68%
Epoch [2/10] - Accuracy: 63.61%
Epoch [3/10] - Accuracy: 66.04%
Epoch [4/10] - Accuracy: 68.96%
Epoch [5/10] - Accuracy: 74.51%
Epoch [6/10] - Accuracy: 76.11%
Epoch [7/10] - Accuracy: 79.93%
Epoch [8/10] - Accuracy: 86.11%
Epoch [9/10] - Accuracy: 87.78%
Epoch [10/10] - Accuracy: 91.32%


In [19]:
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

final_val_ds = MultimodalWheatDataset(val_df, '/content/Kaggle_Prepared')
val_loader = DataLoader(final_val_ds, batch_size=16, shuffle=False)

model_innovative.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for rgb, hs, labels in val_loader:
        rgb, hs = rgb.to(device), hs.to(device)
        outputs = model_innovative(rgb, hs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

print(classification_report(y_true, y_pred, target_names=["Health", "Rust", "Other"]))

              precision    recall  f1-score   support

      Health       0.90      0.90      0.90       120
        Rust       0.97      0.96      0.97       120
       Other       0.90      0.92      0.91       120

    accuracy                           0.93       360
   macro avg       0.93      0.92      0.93       360
weighted avg       0.93      0.93      0.93       360



In [20]:
import tifffile as tiff
import numpy as np
import torch.nn.functional as F # Import for padding

class TripleModalityDataset(Dataset):
    def __init__(self, df, base_dir):
        self.df = df
        self.base_dir = base_dir
        self.label_map = {"Health": 0, "Rust": 1, "Other": 2}
        self.target_hs_channels = 126 # Define a target for HS channels, matching your model

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        split = row['split']
        name = row['file_name']

        # 1. RGB (3 channels)
        rgb_path = os.path.join(self.base_dir, split, 'RGB', name + '.png')
        rgb = cv2.imread(rgb_path)
        rgb = torch.from_numpy(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float() / 255.0

        # 2. MS (5 channels)
        ms_path = os.path.join(self.base_dir, split, 'MS', name + '.tif')
        ms = tiff.imread(ms_path)
        # Ensure ms is 3D (Height, Width, Channels) for consistency before permuting
        if ms.ndim == 2:
            ms = np.expand_dims(ms, axis=-1)
        ms = torch.from_numpy(ms).permute(2, 0, 1).float()

        # 3. HS (126 channels)
        hs_path = os.path.join(self.base_dir, split, 'HS', name + '.tif')
        hs = tiff.imread(hs_path)
        if hs.ndim == 2:
            hs = np.expand_dims(hs, axis=-1)
        hs = torch.from_numpy(hs).permute(2, 0, 1).float()

        # Handle varying HS channel counts
        current_hs_channels = hs.shape[0]
        if current_hs_channels < self.target_hs_channels:
            # Pad with zeros along the channel dimension
            padding_needed = self.target_hs_channels - current_hs_channels
            # pad = (pad_left_width, pad_right_width, pad_left_height, pad_right_height, pad_left_channels, pad_right_channels)
            hs = F.pad(hs, (0, 0, 0, 0, 0, padding_needed), 'constant', 0)
        elif current_hs_channels > self.target_hs_channels:
            # Crop if there are more channels than expected
            hs = hs[:self.target_hs_channels, :, :]

        return rgb, ms, hs, torch.tensor(self.label_map[row['label']])

In [21]:
class TripleFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('resnet50', pretrained=True, num_classes=0)

        self.ms_branch = nn.Sequential(
            nn.Conv2d(5, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        self.hs_branch = nn.Sequential(
            nn.Conv2d(126, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )


        self.classifier = nn.Linear(2048 + 32 + 64, 3)

    def forward(self, rgb, ms, hs):
        f_rgb = self.rgb_branch(rgb)
        f_ms = self.ms_branch(ms)
        f_hs = self.hs_branch(hs)

        combined = torch.cat((f_rgb, f_ms, f_hs), dim=1)
        return self.classifier(combined)

model_final = TripleFusionModel().to(device)
optimizer = torch.optim.Adam(model_final.parameters(), lr=0.0001)

In [23]:
def train_triple_fusion(model, loader, epochs=10):
    model.train()
    print("Train triple fusion  (RGB + MS + HS)...")
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for rgb, ms, hs, labels in loader:
            rgb, ms, hs, labels = rgb.to(device), ms.to(device), hs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(rgb, ms, hs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_acc = 100. * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(loader):.4f} - Accuracy: {epoch_acc:.2f}%")


final_train_ds = TripleModalityDataset(train_df, '/content/Kaggle_Prepared')
final_train_loader = DataLoader(final_train_ds, batch_size=16, shuffle=True)

train_triple_fusion(model_final, final_train_loader, epochs=20)

Train triple fusion  (RGB + MS + HS)...
Epoch [1/20] - Loss: 2.2409 - Accuracy: 68.96%
Epoch [2/20] - Loss: 1.5186 - Accuracy: 74.10%
Epoch [3/20] - Loss: 1.3921 - Accuracy: 73.75%
Epoch [4/20] - Loss: 1.1912 - Accuracy: 77.92%
Epoch [5/20] - Loss: 1.5365 - Accuracy: 77.15%
Epoch [6/20] - Loss: 1.1330 - Accuracy: 80.21%
Epoch [7/20] - Loss: 1.1122 - Accuracy: 80.62%
Epoch [8/20] - Loss: 1.2100 - Accuracy: 80.28%
Epoch [9/20] - Loss: 0.8902 - Accuracy: 84.38%
Epoch [10/20] - Loss: 0.9196 - Accuracy: 85.56%
Epoch [11/20] - Loss: 0.6399 - Accuracy: 87.64%
Epoch [12/20] - Loss: 0.6291 - Accuracy: 85.83%
Epoch [13/20] - Loss: 0.5173 - Accuracy: 88.75%
Epoch [14/20] - Loss: 0.4473 - Accuracy: 90.62%
Epoch [15/20] - Loss: 0.3150 - Accuracy: 92.22%
Epoch [16/20] - Loss: 0.3743 - Accuracy: 91.39%
Epoch [17/20] - Loss: 0.4518 - Accuracy: 90.14%
Epoch [18/20] - Loss: 0.4459 - Accuracy: 89.86%
Epoch [19/20] - Loss: 0.4737 - Accuracy: 91.67%
Epoch [20/20] - Loss: 0.5517 - Accuracy: 90.07%


In [24]:
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

# Define val_loader_triple
final_val_ds = TripleModalityDataset(val_df, '/content/Kaggle_Prepared')
val_loader_triple = DataLoader(final_val_ds, batch_size=16, shuffle=False)

model_final.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for rgb, ms, hs, labels in val_loader_triple:
        rgb, ms, hs = rgb.to(device), ms.to(device), hs.to(device)
        outputs = model_final(rgb, ms, hs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

print(classification_report(y_true, y_pred, target_names=["Health", "Rust", "Other"]))

              precision    recall  f1-score   support

      Health       0.98      0.67      0.79       120
        Rust       0.79      0.98      0.88       120
       Other       0.88      0.94      0.91       120

    accuracy                           0.86       360
   macro avg       0.88      0.86      0.86       360
weighted avg       0.88      0.86      0.86       360



In [25]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

from torch.optim.lr_scheduler import CosineAnnealingLR

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

labels = df['label'].map({"Health": 0, "Rust": 1, "Other": 2}).values

print(f" Fold Cross-Validation on triple fusion ")

for fold, (train_idx, val_idx) in enumerate(skf.split(df, labels)):
    print(f"\n#️ Fold {fold+1}/5")

    train_df_fold = df.iloc[train_idx]
    val_df_fold = df.iloc[val_idx]

    train_ds = TripleModalityDataset(train_df_fold, '/content/Kaggle_Prepared')
    val_ds = TripleModalityDataset(val_df_fold, '/content/Kaggle_Prepared')

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    model_fold = TripleFusionModel().to(device)
    optimizer = torch.optim.Adam(model_fold.parameters(), lr=0.0001)
    criterion = nn.CrossEntropyLoss()



    scheduler = CosineAnnealingLR(optimizer, T_max=20) # لـ 20 Epochs

    for epoch in range(20):
        model_fold.train()
        running_loss = 0.0
        for rgb, ms, hs, target in train_loader:
            rgb, ms, hs, target = rgb.to(device), ms.to(device), hs.to(device), target.to(device)

            optimizer.zero_grad()
            output = model_fold(rgb, ms, hs)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/20 - Loss: {running_loss/len(train_loader):.4f}")

    model_fold.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for rgb, ms, hs, target in val_loader:
            rgb, ms, hs, target = rgb.to(device), ms.to(device), hs.to(device), target.to(device)
            output = model_fold(rgb, ms, hs)
            _, pred = torch.max(output, 1)
            total += target.size(0)
            correct += (pred == target).sum().item()

    fold_acc = 100. * correct / total
    fold_results.append(fold_acc)
    print(f"Fold {fold+1} Accuracy: {fold_acc:.2f}%")

print(f"\5 Folds mean accuracy: {np.mean(fold_results):.2f}% (+/- {np.std(fold_results):.2f})")

 Fold Cross-Validation on triple fusion 

#️ Fold 1/5
Epoch 1/20 - Loss: 7.6232
Epoch 2/20 - Loss: 5.3233
Epoch 3/20 - Loss: 5.4658
Epoch 4/20 - Loss: 4.1931
Epoch 5/20 - Loss: 3.5390
Epoch 6/20 - Loss: 3.0252
Epoch 7/20 - Loss: 3.0636
Epoch 8/20 - Loss: 2.4231
Epoch 9/20 - Loss: 1.5968
Epoch 10/20 - Loss: 2.0620
Epoch 11/20 - Loss: 1.9038
Epoch 12/20 - Loss: 1.3986
Epoch 13/20 - Loss: 0.7271
Epoch 14/20 - Loss: 0.7385
Epoch 15/20 - Loss: 0.4659
Epoch 16/20 - Loss: 0.3955
Epoch 17/20 - Loss: 0.3462
Epoch 18/20 - Loss: 0.3288
Epoch 19/20 - Loss: 0.2964
Epoch 20/20 - Loss: 0.2829
Fold 1 Accuracy: 85.83%

#️ Fold 2/5
Epoch 1/20 - Loss: 8.3790
Epoch 2/20 - Loss: 5.6990
Epoch 3/20 - Loss: 5.3626
Epoch 4/20 - Loss: 5.6809
Epoch 5/20 - Loss: 3.4610
Epoch 6/20 - Loss: 2.6433
Epoch 7/20 - Loss: 2.2793
Epoch 8/20 - Loss: 2.1022
Epoch 9/20 - Loss: 2.0141
Epoch 10/20 - Loss: 1.6625
Epoch 11/20 - Loss: 2.0198
Epoch 12/20 - Loss: 0.8369
Epoch 13/20 - Loss: 0.8793
Epoch 14/20 - Loss: 0.6762
Epoch 15/

In [26]:
torch.save(model_fold.state_dict(), 'Wheat_TripleFusion_Final_Model.pth')